# 03 Baseline Models

This notebook trains baseline regressors and saves MAE/RMSE metrics for comparison against CNN.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from src.evaluate import evaluate_predictions, save_metrics

root_dir = Path.cwd()
processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

candidate_files = [
    processed_dir / "desharnais_processed.csv",
    processed_dir / "cocomo81_processed.csv",
    processed_dir / "china_processed.csv",
]

dataset_path = next((p for p in candidate_files if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create processed datasets")

df = pd.read_csv(dataset_path)
target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Using dataset:", dataset_path.name)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

try:
    from xgboost import XGBRegressor
    models["xgboost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )
except ImportError:
    print("xgboost is not installed. Install with: pip install xgboost")

In [ ]:
metrics_frames = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    row = evaluate_predictions(name, y_test, preds)
    metrics_frames.append(row)

baseline_metrics = pd.concat(metrics_frames, ignore_index=True)
baseline_metrics = baseline_metrics.sort_values("rmse")
baseline_metrics

In [ ]:
metrics_path = save_metrics(
    baseline_metrics,
    file_name="baseline_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved baseline metrics to:", metrics_path)